In [0]:
# =============================================================================
# jobs/25_warning_history_periods.py
# When was each flood alert, flood warning and severe flood warning in force, and for
# how long? Built from a file of every message the EA has issued, joined to
# flood_warnings_log (job 11), which carries on from when polling began.
#
# What this script does:
#   1. Gets the message history: one row per message with the area code (TACode), time
#      (dateTime) and severity ("Flood alert", "Flood warning", "Severe flood warning" or
#      "Warning no longer in force").
#        - Given a source_path (a parquet file, or a folder of them, in a Databricks volume),
#          it reads that file, checks it, and stores the clean messages in the Delta table
#          ea_warning_messages_history. Do this ONCE: the file is a closed record and does not change.
#        - Given no source_path it reads ea_warning_messages_history, so a scheduled run
#          needs no file and no path. New data arrives only through flood_warnings_log.
#   1b. Reads flood_warnings_log, the live record, and turns it into the same kind of message
#      (a change to level 1 to 3 is a message at that level, level 4 is "no longer in force",
#      and an area that left the feed without a 4 is switched off when the poll noticed:
#      ended_by = dropped_from_feed). The two are joined and an exact repeat is kept once.
#   2. Replays each area's messages in time order with the rules in
#      utils/warning_history.py:
#        Flood alert                  switches an alert on.
#        Flood warning                switches a warning on, or turns a severe flood
#                                     warning down one level.
#        Severe flood warning         raises an existing warning to severe.
#        Warning no longer in force   switches the area off, from any level.
#      A message at the level an area is already at is an update, not a change.
#   3. Writes two tables, each with start, end and duration:
#        ea_warning_episodes  one row per run in force at ANY level. A warning that goes
#                             severe and comes back down is one episode.
#        ea_warning_periods   one row per run at ONE level inside an episode, so that
#                             warning has a flood warning period, a severe period and a
#                             flood warning period again.
#   4. Overwrites both tables in full: the file is the whole record, so a re-run gives
#      the same answer. It never changes anything outside these two tables.
#
# THE GAP. The file ends before the log's first poll. An area on when the file ends stays on
# until the log says otherwise, and a warning that started and ended inside that stretch is
# in neither source. Episodes that cross it are marked spans_gap: treat their duration with
# care. start_origin and end_origin say which source each start and end came from.
#
# TIMES. The times are used as written in the file (dateTime, which carries a "Z"). If the
# source recorded UK clock time and labelled it Z, a duration that spans a clock change
# (late March or late October) is out by an hour. Durations inside one day are not affected.
#
# OPEN EPISODES. Anything with no "no longer in force" by the end of the file is open: it
# has no end_time, its duration is measured to the latest message in the file, and
# stale_open is true when that is over 30 days (a "no longer in force" is probably missing).
#
# Parameters (widgets):
#   source_path   OPTIONAL. The parquet file or folder, for example
#                 /Volumes/<catalog>/<schema>/<volume>/2026_06_08_All_Messages_FWIS.parquet
#                 Set it for the first run (or to reload the file); leave it blank after that.
#   include_log   "true" (default) joins flood_warnings_log; "false" uses the history alone
#   skip_if_fresh_minutes   0 (default) always rebuilds. Master_polling passes 60: the run is
#                 skipped (exit "skipped_fresh") when the tables were built less than that long
#                 ago AND flood_warnings_log has no poll newer than the one the build used. The log
#                 only gains rows when something changes, so quiet polls cost nothing, and the age
#                 limit still rebuilds hourly so the open episodes' durations keep growing.
#
# SCHEDULE. Master_polling runs this on every poll with skip_if_fresh_minutes = 60 and no
# source_path, as a non-fatal step: a failure is reported but does not fail the polling run.
# OPEN DURATIONS are measured to the latest poll (flood_warnings_current), not just the latest
# message, so a warning still in force keeps growing until it ends.
#
# Exit values: "success".
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, TimestampType
)
import time as _time
_started = _time.time()

from config import (
    TBL_WARNING_EPISODES, TBL_WARNING_PERIODS, TBL_WARNING_MESSAGES_HISTORY, TBL_FLOOD_WARN_LOG,
    TBL_FLOOD_WARN_CURRENT
)
from utils.helpers import get_spark, utc_now, table_exists
from utils.warning_history import (
    normalise_messages, build_history, events_from_log, combine_events, should_skip_rebuild,
    ORIGIN_FILE, KIND_MESSAGE
)

dbutils.widgets.text("source_path", "", "Parquet file or folder (first run only)")
dbutils.widgets.text("include_log", "true", "Join flood_warnings_log (true or false)")
dbutils.widgets.text("skip_if_fresh_minutes", "0", "Skip if built this recently and the log is unchanged (0 = never)")
source_path = dbutils.widgets.get("source_path").strip()
include_log = dbutils.widgets.get("include_log").strip().lower() != "false"
skip_fresh = int(dbutils.widgets.get("skip_if_fresh_minutes").strip() or 0)

spark   = get_spark()
now_iso = utc_now().isoformat()

if not source_path and not table_exists(spark, TBL_WARNING_MESSAGES_HISTORY):
    raise Exception(
        f"{TBL_WARNING_MESSAGES_HISTORY} does not exist yet. Run this notebook once with the "
        f"source_path widget set to the parquet file holding the message history, for example "
        f"/Volumes/<catalog>/<schema>/<volume>/<file>.parquet. It stores the messages there, and "
        f"later runs need no path."
    )

# Skip a scheduled rebuild when nothing has changed (see skip_if_fresh_minutes above).
log_latest = None
if include_log and table_exists(spark, TBL_FLOOD_WARN_LOG):
    log_latest = spark.sql(f"SELECT MAX(ingested_at) AS m FROM {TBL_FLOOD_WARN_LOG}").collect()[0]["m"]

skip_rebuild = False
if skip_fresh > 0 and not source_path and table_exists(spark, TBL_WARNING_EPISODES):
    try:
        prev = spark.sql(f"SELECT MAX(loaded_at) AS built, MAX(log_through) AS through "
                         f"FROM {TBL_WARNING_EPISODES}").collect()[0]
        built = pd.Timestamp(prev["built"]).tz_convert(None) if prev["built"] else None
        through = pd.Timestamp(prev["through"]) if prev["through"] is not None else None
        latest = pd.Timestamp(log_latest) if log_latest is not None else None
        skip_rebuild = should_skip_rebuild(built, through, latest, pd.Timestamp(utc_now()).tz_convert(None), skip_fresh)
        if skip_rebuild:
            print(f"Tables built {built} (less than {skip_fresh} minutes ago) and the log has nothing newer "
                  f"than {through}. Skipping.")
    except Exception as e:
        print(f"Could not check whether a rebuild is needed ({e}); rebuilding.")

if skip_rebuild:
    dbutils.notebook.exit("skipped_fresh")


In [0]:
# =============================================================================
# GET THE MESSAGE HISTORY
# From the file (first run) or from the table that holds it (every run after). pandas, not
# Spark, reads the file, because it holds two columns called "severity" and "Severity":
# Spark treats those as the same name, pandas does not.
# =============================================================================

if source_path:
    raw = pd.read_parquet(source_path)
    print(f"Read {len(raw):,} messages from {source_path}")
    print(f"Columns: {list(raw.columns)}")

    events, report = normalise_messages(raw)
    print(f"\nUsable messages: {len(events):,} for {events['ea_area_code'].nunique():,} areas, "
          f"{events['time'].min()} to {events['time'].max()}")
    print(f"  no area code: {report['no_code']}   no readable time: {report['no_time']}   "
          f"unrecognised severity: {report['unknown_severity']}   exact repeats dropped: {report['duplicates']}")
    if report["unknown_values"]:
        print(f"  severity values not recognised: {report['unknown_values']}")
    if report["disagreeing_levels"]:
        print(f"  WARNING: {report['disagreeing_levels']} rows where the text and the numeric severity disagree "
              f"(the text was used).")

    # Store the clean messages once, so scheduled runs need neither the file nor its path.
    HISTORY_SCHEMA = StructType([
        StructField("ea_area_code", StringType(),    True),
        StructField("time",         TimestampType(), True),
        StructField("level",        StringType(),    True),
        StructField("text",         StringType(),    True),
        StructField("source",       StringType(),    True),   # FWS or History Log, as in the file
        StructField("loaded_at",    StringType(),    True),
    ])
    stored = [(str(r.ea_area_code), r.time.to_pydatetime(), str(r.level),
               None if pd.isna(r.text) else str(r.text), None if pd.isna(r.source) else str(r.source), now_iso)
              for r in events.itertuples(index=False)]
    (
        spark.createDataFrame(stored, HISTORY_SCHEMA).write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TBL_WARNING_MESSAGES_HISTORY)
    )
    spark.sql(f"""
        COMMENT ON TABLE {TBL_WARNING_MESSAGES_HISTORY} IS
        'Every EA flood alert and warning message in the history file, cleaned (area code, time, level), loaded once by 25_warning_history_periods. The file is a closed record; new messages come from flood_warnings_log.'
    """)
    print(f"Stored {len(stored):,} messages in {TBL_WARNING_MESSAGES_HISTORY}")
else:
    stored_events = spark.table(TBL_WARNING_MESSAGES_HISTORY).toPandas()
    events = stored_events[["ea_area_code", "time", "level", "text", "source"]].copy()
    events["time"] = pd.to_datetime(events["time"])
    events["origin"], events["kind"] = ORIGIN_FILE, KIND_MESSAGE
    loaded = stored_events["loaded_at"].max()
    print(f"Read {len(events):,} history messages for {events['ea_area_code'].nunique():,} areas from "
          f"{TBL_WARNING_MESSAGES_HISTORY} (loaded {loaded}), {events['time'].min()} to {events['time'].max()}")

print("\nMessages by severity:")
print(events["level"].value_counts().to_string())

# =============================================================================
# THE LIVE LOG, JOINED TO THE FILE
# =============================================================================

gap = None
last_poll = None
if table_exists(spark, TBL_FLOOD_WARN_CURRENT):
    last_poll = spark.sql(f"SELECT MAX(ingested_at) AS m FROM {TBL_FLOOD_WARN_CURRENT}").collect()[0]["m"]
if include_log and table_exists(spark, TBL_FLOOD_WARN_LOG):
    log = (spark.table(TBL_FLOOD_WARN_LOG)
           .select("flood_area_id", "change_type", "severity_level", "time_severity_changed",
                   "time_message_changed", "ingested_at")
           .toPandas())
    log_events, log_report = events_from_log(log)
    file_end = events["time"].max()
    first_poll = pd.to_datetime(log["ingested_at"], utc=True).dt.tz_localize(None).min()
    print(f"\nflood_warnings_log: {log_report['rows']:,} rows, {len(log_events):,} usable events for "
          f"{log_events['ea_area_code'].nunique():,} areas; first poll {first_poll}")
    print(f"  unusable: no code {log_report['no_code']}, no time {log_report['no_time']}, "
          f"unknown level {log_report['unknown_level']}.   Switch-offs inferred from an area leaving "
          f"the feed with no level 4: {log_report['closed_fallback']}")
    events, repeats = combine_events(events, log_events)
    print(f"  Messages in both sources, kept once: {repeats:,}")
    if first_poll > file_end:
        gap = (file_end, first_poll)
        print(f"  GAP: the file ends {file_end} and the log's first poll is {first_poll}. "
              f"Episodes crossing it are marked spans_gap.")
    else:
        print(f"  The sources overlap: the file ends {file_end}, the log starts {first_poll}.")
elif include_log:
    print(f"\n{TBL_FLOOD_WARN_LOG} does not exist, so only the file is used. Run job 11 first to add the log.")
else:
    print("\ninclude_log is false: using the file alone.")


In [0]:
# =============================================================================
# BUILD THE EPISODES AND PERIODS, AND WRITE THEM
# =============================================================================

# Open episodes are measured to the latest poll, which is later than the latest message when
# nothing has changed for a while.
as_of = events["time"].max()
if include_log and last_poll is not None:
    as_of = max(as_of, pd.Timestamp(last_poll).tz_localize(None) if pd.Timestamp(last_poll).tzinfo else pd.Timestamp(last_poll))
episodes, periods, stats = build_history(events, as_of=as_of, gap=gap)

print(f"\nEpisodes: {len(episodes):,}   periods: {len(periods):,}   "
      f"'no longer in force' with nothing on: {stats['orphan_off']}")
print("\nEpisodes by area type, with the median and longest duration in hours:")
closed = episodes[~episodes["is_open"]]
print(closed.groupby("area_type")["duration_hours"].agg(["count", "median", "max"]).round(1).to_string())
n_open = int(episodes["is_open"].sum())
n_stale = int(episodes["stale_open"].sum())
print(f"\nStill in force at the end of the file: {n_open}   (open for over 30 days, probably missing their end: {n_stale})")
print(f"Episodes that went severe: {int(episodes['had_severe'].sum())}")
print(f"Ended by the feed dropping the area, with no level 4 seen: {int((episodes['ended_by'] == 'dropped_from_feed').sum())}")
print(f"Crossing the gap between the file and the log: {int(episodes['spans_gap'].sum())}")
print("\nWhere episodes began and ended:")
print(pd.crosstab(episodes["start_origin"].fillna("-"), episodes["end_origin"].fillna("still open")).to_string())

EPISODE_SCHEMA = StructType([
    StructField("episode_id",       StringType(),    True),
    StructField("ea_area_code",     StringType(),    True),
    StructField("area_type",        StringType(),    True),   # flood_alert or flood_warning, from the area code
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),   # null while open
    StructField("duration_minutes", DoubleType(),    True),   # to the end of the file while open
    StructField("duration_hours",   DoubleType(),    True),
    StructField("start_level",      StringType(),    True),
    StructField("peak_level",       StringType(),    True),
    StructField("had_severe",       BooleanType(),   True),
    StructField("severe_minutes",   DoubleType(),    True),   # time at severe inside the episode
    StructField("n_severe_periods", IntegerType(),   True),
    StructField("n_messages",       IntegerType(),   True),   # messages while in force, updates included
    StructField("ended_by",         StringType(),    True),   # no_longer_in_force, dropped_from_feed or open
    StructField("is_open",          BooleanType(),   True),
    StructField("stale_open",       BooleanType(),   True),
    StructField("start_origin",     StringType(),    True),   # history_file or flood_warnings_log
    StructField("end_origin",       StringType(),    True),
    StructField("spans_gap",        BooleanType(),   True),   # crosses the stretch neither source covers
    StructField("data_as_of",       TimestampType(), True),   # the latest poll or message used
    StructField("log_through",      TimestampType(), True),   # the latest log poll this build used
    StructField("loaded_at",        StringType(),    True),
])

PERIOD_SCHEMA = StructType([
    StructField("period_id",        StringType(),    True),
    StructField("episode_id",       StringType(),    True),
    StructField("ea_area_code",     StringType(),    True),
    StructField("level",            StringType(),    True),   # flood_alert, flood_warning or severe_flood_warning
    StructField("start_time",       TimestampType(), True),
    StructField("end_time",         TimestampType(), True),
    StructField("duration_minutes", DoubleType(),    True),
    StructField("duration_hours",   DoubleType(),    True),
    StructField("started_by",       StringType(),    True),   # issued, upgraded or downgraded
    StructField("ended_by",         StringType(),    True),   # upgraded, downgraded, no_longer_in_force or open
    StructField("is_open",          BooleanType(),   True),
    StructField("start_origin",     StringType(),    True),
    StructField("data_as_of",       TimestampType(), True),
    StructField("loaded_at",        StringType(),    True),
])


def to_rows(frame, schema):
    """Rows as tuples in schema order, with pandas NaT and NaN as None."""
    frame = frame.copy()
    frame["data_as_of"] = as_of
    frame["log_through"] = pd.Timestamp(log_latest) if log_latest is not None else pd.NaT
    frame["loaded_at"] = now_iso
    rows = []
    for rec in frame[[f.name for f in schema.fields]].itertuples(index=False, name=None):
        row = []
        for value, field in zip(rec, schema.fields):
            if value is None or (not isinstance(value, (bool, str)) and pd.isna(value)):
                row.append(None)
            elif isinstance(field.dataType, TimestampType):
                row.append(pd.Timestamp(value).to_pydatetime())
            elif isinstance(field.dataType, BooleanType):
                row.append(bool(value))
            elif isinstance(field.dataType, IntegerType):
                row.append(int(value))
            elif isinstance(field.dataType, DoubleType):
                row.append(float(value))
            else:
                row.append(str(value))
        rows.append(tuple(row))
    return rows


for table, frame, schema, comment in (
    (TBL_WARNING_EPISODES, episodes, EPISODE_SCHEMA,
     "Each run of a flood alert, flood warning or severe flood warning being in force in an area, "
     "from the message that switched it on to the Warning no longer in force that ended it, with start, "
     "end and duration. A warning that goes severe and back is one episode. Built by 25_warning_history_periods "
     "from the EA message history file and flood_warnings_log. Times are as written in the source. is_open means "
     "no end by the end of the data. spans_gap marks episodes crossing the stretch between the file and the log. "
     "ended_by dropped_from_feed means the area left the live feed with no Warning no longer in force seen."),
    (TBL_WARNING_PERIODS, periods, PERIOD_SCHEMA,
     "Each run at ONE level (flood alert, flood warning or severe flood warning) inside an episode in "
     "ea_warning_episodes, with start, end and duration. Built by 25_warning_history_periods from the message "
     "history file and flood_warnings_log."),
):
    (
        spark.createDataFrame(to_rows(frame, schema), schema).write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table)
    )
    spark.sql(f"COMMENT ON TABLE {table} IS '{comment}'")
    print(f"Wrote {spark.table(table).count():,} rows to {table}")

print(f"\nDone in {_time.time() - _started:,.0f} seconds.")
dbutils.notebook.exit("success")